# 29 — Discovery Specifies Retrieval

**Leading specification:** retrieval begins by mapping a content identity to candidate replica locations.

Notebook 23 established that verified chunks become more available when they exist at independent replicas:

\[
L_i=\{\ell_{i,1},\ell_{i,2},\ldots,\ell_{i,R_i}\}
\]

Notebook 29 asks the next question:

> How does a client find those replicas?

Discovery answers by turning a content ID into a candidate set. Verification still decides trust after retrieval.

## 1. Context

By this point the repository has introduced:

```text
content identity
→ verification
→ chunking
→ replication
```

Discovery adds a lookup layer:

```text
content ID
→ lookup
→ candidate replicas
→ select source
→ download bytes
→ verify
```

Discovery answers:

> Where can I get it?

Verification answers:

> Is it correct?

The two should not be confused.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "29_discovery"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Discovery variables

A discovery function maps a content identity to candidate locations:

\[
D(I)\rightarrow\{L_1,L_2,\ldots,L_n\}
\]

The client chooses a subset or source:

\[
S\subseteq D(I)
\]

Retrieval is complete only after verification:

\[
T = D(I)\rightarrow S\rightarrow C_{\mathrm{received}}\rightarrow V
\]

Discovery is about reachability. Verification is about correctness.

In [ ]:
@dataclass(frozen=True)
class DiscoveryVariables:
    content_identity: str
    discovery: str
    candidates: str
    selection: str
    verification: str

variables = DiscoveryVariables(
    content_identity="I = content ID or digest requested by the client",
    discovery="D(I) = lookup operation returning candidate locations",
    candidates="{L_1,...,L_n} = replicas that claim to hold I",
    selection="S subset of D(I), the source or sources attempted by the client",
    verification="V checks whether received bytes hash back to I",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Create a replicated object

Notebook 29 uses one deterministic object and several candidate replica records. Some are reachable, one is down, and one is malicious or stale.

In [ ]:
artifact_dir = RESULTS / "artifacts"
replica_root = RESULTS / "replicas"
retrieval_dir = RESULTS / "retrieval"

for d in [artifact_dir, replica_root, retrieval_dir]:
    d.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "discovery-object.bin"
payload = (
    b"open-model-distribution discovery notebook\n"
    + b"lookup and retrieval payload\n"
    + bytes(range(256)) * 24
)
artifact_path.write_bytes(payload)

identity = content_id(artifact_path)
digest = sha256_file(artifact_path)

replica_specs = [
    {"replica": "server_A", "kind": "server", "reachable": True, "valid": True, "latency_ms": 52},
    {"replica": "mirror_B", "kind": "mirror", "reachable": False, "valid": True, "latency_ms": None},
    {"replica": "peer_C", "kind": "peer", "reachable": True, "valid": True, "latency_ms": 88},
    {"replica": "archive_D", "kind": "archive", "reachable": True, "valid": True, "latency_ms": 140},
    {"replica": "cache_E", "kind": "cache", "reachable": True, "valid": False, "latency_ms": 35},
]

rows = []
for spec in replica_specs:
    rdir = replica_root / spec["replica"]
    rdir.mkdir(parents=True, exist_ok=True)
    path = rdir / "discovery-object.bin"

    if spec["valid"]:
        shutil.copyfile(artifact_path, path)
    else:
        altered = bytearray(payload)
        altered[99] = (altered[99] + 1) % 256
        path.write_bytes(bytes(altered))

    rows.append({
        "content_id": identity,
        "replica": spec["replica"],
        "kind": spec["kind"],
        "reachable": spec["reachable"],
        "valid_bytes": spec["valid"],
        "latency_ms": spec["latency_ms"],
        "path": str(path.relative_to(REPO_ROOT)),
        "digest": sha256_file(path),
        "matches_identity": sha256_file(path) == digest,
    })

replicas_df = pd.DataFrame(rows)
replicas_csv = RESULTS / "29_replica_candidates.csv"
replicas_df.to_csv(replicas_csv, index=False)

object_identity = {
    "object": str(artifact_path.relative_to(REPO_ROOT)),
    "content_id": identity,
    "digest": digest,
    "size_bytes": artifact_path.stat().st_size,
}

object_identity_path = RESULTS / "29_object_identity.json"
object_identity_path.write_text(json.dumps(object_identity, indent=2), encoding="utf-8")

replicas_df

## 4. Discovery manifest

The discovery manifest records candidate locations for a content identity. It does not prove correctness by itself.

\[
D(I)\rightarrow\{L_1,\ldots,L_n\}
\]

In [ ]:
discovery_manifest = {
    "schema": "open-model-distribution.discovery-manifest.v0",
    "content_id": identity,
    "digest": digest,
    "candidate_count": len(replicas_df),
    "candidates": replicas_df[
        ["replica", "kind", "reachable", "latency_ms", "path"]
    ].to_dict(orient="records"),
    "statement": "Discovery returns candidate locations; verification decides correctness.",
}

discovery_manifest_path = RESULTS / "29_discovery_manifest.json"
discovery_manifest_path.write_text(json.dumps(discovery_manifest, indent=2), encoding="utf-8")

discovery_manifest

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.axis("off")
ax.set_title("Discovery pipeline", fontsize=17, pad=18)

nodes = [
    ("content\nID", 0.10),
    ("lookup", 0.28),
    ("candidate\nreplicas", 0.48),
    ("select\nsource", 0.68),
    ("download\n+ verify", 0.88),
]

for label, x in nodes:
    ax.text(x, 0.58, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.3),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.06, 0.58), xytext=(x1 + 0.06, 0.58),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.20, "discovery finds candidates; verification accepts bytes", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
discovery_pipeline_fig = FIGURES / "29_discovery_pipeline.png"
fig.savefig(discovery_pipeline_fig, dpi=180)
plt.show()

discovery_pipeline_fig

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")
ax.set_title("Lookup manifest", fontsize=17, pad=16)

ax.text(0.18, 0.70, "content ID", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.22, 0.70, identity[:32] + "…", ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.62, 0.78, "candidate replicas", ha="center", va="center", fontsize=13, weight="bold", transform=ax.transAxes)

y = 0.64
for _, row in replicas_df.iterrows():
    status = "up" if row["reachable"] else "down"
    ax.text(0.44, y, row["replica"], ha="left", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.20", fc="white", ec="black", lw=1.0),
            transform=ax.transAxes)
    ax.text(0.64, y, f"{row['kind']} · {status}", ha="left", va="center", fontsize=11, transform=ax.transAxes)
    y -= 0.10

fig.tight_layout()
lookup_manifest_fig = FIGURES / "29_lookup_manifest.png"
fig.savefig(lookup_manifest_fig, dpi=180)
plt.show()

lookup_manifest_fig

## 5. Discovery graph

A content identity can point to many candidate locations. The graph shows candidate reachability, not correctness.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")
ax.set_title("Discovery graph", fontsize=17, pad=18)

ax.text(0.16, 0.52, "content ID\n" + digest[:12] + "…", ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.42", fc="white", ec="black", lw=1.4),
        transform=ax.transAxes)

positions = [(0.52, 0.82), (0.72, 0.66), (0.75, 0.42), (0.58, 0.24), (0.40, 0.36)]
for (_, row), (x, y) in zip(replicas_df.iterrows(), positions):
    status = "reachable" if row["reachable"] else "down"
    ax.text(x, y, f"{row['replica']}\n{status}", ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    ax.annotate("", xy=(x - 0.09, y), xytext=(0.25, 0.52),
                arrowprops=dict(arrowstyle="->", lw=1.7), xycoords=ax.transAxes)

ax.text(0.5, 0.08, "lookup returns candidates, including unreachable or invalid sources", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
discovery_graph_fig = FIGURES / "29_discovery_graph.png"
fig.savefig(discovery_graph_fig, dpi=180)
plt.show()

discovery_graph_fig

## 6. Select a candidate and verify

A discovery result may contain:

- unreachable replicas
- valid reachable replicas
- reachable replicas with invalid bytes

The client selects a reachable candidate, downloads bytes, and verifies them. If verification fails, it tries another candidate.

In [ ]:
def attempt_retrieval(candidate_row):
    record = {
        "replica": candidate_row["replica"],
        "kind": candidate_row["kind"],
        "reachable": bool(candidate_row["reachable"]),
        "path": candidate_row["path"],
        "attempted": False,
        "downloaded": False,
        "verification": "SKIP",
        "accepted": False,
        "reason": "",
    }

    if not candidate_row["reachable"]:
        record["reason"] = "replica unreachable"
        return record

    record["attempted"] = True
    src = REPO_ROOT / candidate_row["path"]
    dst = retrieval_dir / f"retrieved_from_{candidate_row['replica']}.bin"
    shutil.copyfile(src, dst)
    record["downloaded"] = True
    record["retrieved_path"] = str(dst.relative_to(REPO_ROOT))
    local_digest = sha256_file(dst)
    record["local_digest"] = local_digest

    if local_digest == digest:
        record["verification"] = "PASS"
        record["accepted"] = True
        record["reason"] = "verified content identity"
    else:
        record["verification"] = "FAIL"
        record["accepted"] = False
        record["reason"] = "digest mismatch"

    return record

# Try candidates ordered by latency when reachable, with unreachable kept in the result set.
ordered = replicas_df.copy()
ordered["latency_sort"] = ordered["latency_ms"].fillna(10**9)
ordered = ordered.sort_values(["reachable", "latency_sort"], ascending=[False, True])

attempts = []
accepted_record = None
for _, candidate in ordered.iterrows():
    record = attempt_retrieval(candidate)
    attempts.append(record)
    if record["accepted"]:
        accepted_record = record
        break

retrieval_df = pd.DataFrame(attempts)
retrieval_csv = RESULTS / "29_retrieval_attempts.csv"
retrieval_df.to_csv(retrieval_csv, index=False)

retrieval_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axis("off")
ax.set_title("Lookup versus verification", fontsize=17, pad=16)

left = [("Discovery", "Where can I get it?", 0.25), ("Verification", "Is it correct?", 0.72)]
for title, subtitle, x in left:
    ax.text(x, 0.66, title, ha="center", va="center", fontsize=15, weight="bold", transform=ax.transAxes)
    ax.text(x, 0.48, subtitle, ha="center", va="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.40", fc="white", ec="black", lw=1.4),
            transform=ax.transAxes)

ax.annotate("", xy=(0.58, 0.48), xytext=(0.39, 0.48),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.18, "candidate lists do not replace digest checks", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
lookup_vs_verification_fig = FIGURES / "29_lookup_vs_verification.png"
fig.savefig(lookup_vs_verification_fig, dpi=180)
plt.show()

lookup_vs_verification_fig

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axis("off")
ax.set_title("Discovery flow", fontsize=17, pad=16)

steps = [
    ("request\ncontent ID", 0.10),
    ("lookup\nmanifest", 0.27),
    ("candidate\nlist", 0.44),
    ("try\nsource", 0.61),
    ("download", 0.76),
    ("verify", 0.90),
]

for label, x in steps:
    ax.text(x, 0.58, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(steps[:-1], steps[1:]):
    ax.annotate("", xy=(x2 - 0.045, 0.58), xytext=(x1 + 0.045, 0.58),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

if accepted_record:
    footer = f"accepted: {accepted_record['replica']} after local verification"
else:
    footer = "no candidate verified"

ax.text(0.5, 0.22, footer, ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
discovery_flow_fig = FIGURES / "29_discovery_flow.png"
fig.savefig(discovery_flow_fig, dpi=180)
plt.show()

discovery_flow_fig

## 7. Failed lookup paths

Discovery can produce unusable candidates. A robust client treats lookup results as candidates, not truth.

In [ ]:
candidate_status = replicas_df.copy()
candidate_status["candidate_status"] = candidate_status.apply(
    lambda row: "unreachable" if not row["reachable"] else ("valid" if row["matches_identity"] else "invalid bytes"),
    axis=1,
)

candidate_status_csv = RESULTS / "29_candidate_status.csv"
candidate_status.to_csv(candidate_status_csv, index=False)

candidate_status[["replica", "kind", "candidate_status", "matches_identity"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
values = candidate_status["candidate_status"].map({"valid": 1, "invalid bytes": 0, "unreachable": -1})
ax.bar(candidate_status["replica"], values)
ax.set_ylim(-1.3, 1.3)
ax.set_ylabel("candidate status")
ax.set_yticks([-1, 0, 1])
ax.set_yticklabels(["unreachable", "invalid", "valid"])
ax.set_title("Discovery results can include failed candidates")

for idx, row in candidate_status.iterrows():
    ax.text(idx, values.iloc[idx] + (0.08 if values.iloc[idx] >= 0 else -0.18),
            row["candidate_status"], ha="center", va="bottom" if values.iloc[idx] >= 0 else "top", fontsize=9)

fig.tight_layout()
failed_lookup_fig = FIGURES / "29_failed_lookup.png"
fig.savefig(failed_lookup_fig, dpi=180)
plt.show()

failed_lookup_fig

## 8. Retrieval report

The retrieval report records the selected source and the verification decision.

In [ ]:
retrieval_report = {
    "schema": "open-model-distribution.discovery-report.v0",
    "content_id": identity,
    "candidate_count": len(replicas_df),
    "attempt_count": len(retrieval_df),
    "accepted": accepted_record is not None,
    "accepted_replica": accepted_record["replica"] if accepted_record else None,
    "accepted_reason": accepted_record["reason"] if accepted_record else None,
    "statement": "Discovery finds candidates; verification accepts bytes.",
    "attempts": retrieval_df.to_dict(orient="records"),
}

retrieval_report_path = RESULTS / "29_retrieval_report.json"
retrieval_report_path.write_text(json.dumps(retrieval_report, indent=2), encoding="utf-8")

retrieval_report

## 9. Engineering summary

| Property | Specification | Notebook role |
|---|---|---|
| Lookup | \(D(I)\) | Maps a content identity to candidate replica locations. |
| Candidate set | \(\{L_1,\ldots,L_n\}\) | Lists where bytes may be retrieved. |
| Selection | \(S\subseteq D(I)\) | Chooses one or more sources to attempt. |
| Verification | Local digest check | Accepts only bytes matching the content identity. |
| Failure handling | Try another candidate | Unreachable or invalid candidates do not invalidate the identity. |

## 10. Engineering statement

Discovery specifies retrieval by mapping a content identity to candidate replica locations. Discovery does not establish correctness; it only provides possible sources. The receiver still verifies the downloaded bytes before accepting them.

## 11. Key equations

Discovery lookup:

\[
D(I)\rightarrow\{L_1,L_2,\ldots,L_n\}
\]

Candidate selection:

\[
S\subseteq D(I)
\]

Retrieval with verification:

\[
T=D(I)\rightarrow S\rightarrow C_{\mathrm{received}}\rightarrow V
\]

Verification condition:

\[
V=\bigl(H(C_{\mathrm{received}})=I\bigr)
\]

Availability with replicas and paths:

\[
A\propto RP
\]

## 12. Notebook outputs

Notebook 29 writes:

- `results/29_discovery/29_object_identity.json`
- `results/29_discovery/29_discovery_manifest.json`
- `results/29_discovery/29_replica_candidates.csv`
- `results/29_discovery/29_retrieval_attempts.csv`
- `results/29_discovery/29_candidate_status.csv`
- `results/29_discovery/29_retrieval_report.json`
- `figures/29_discovery_pipeline.png`
- `figures/29_lookup_manifest.png`
- `figures/29_discovery_graph.png`
- `figures/29_lookup_vs_verification.png`
- `figures/29_discovery_flow.png`
- `figures/29_failed_lookup.png`

In [ ]:
notebook_manifest = {
    "notebook": "29_discovery.ipynb",
    "title": "Discovery Specifies Retrieval",
    "primary_specification": "discovery",
    "statement": "Discovery maps a content identity to candidate replica locations; verification accepts bytes.",
    "equations": [
        "D(I)->{L_1,...,L_n}",
        "S subset of D(I)",
        "T=D(I)->S->C_received->V",
        "V=(H(C_received)=I)",
        "A proportional to R P",
    ],
    "outputs": {
        "object_identity": str(object_identity_path.relative_to(REPO_ROOT)),
        "discovery_manifest": str(discovery_manifest_path.relative_to(REPO_ROOT)),
        "replica_candidates_csv": str(replicas_csv.relative_to(REPO_ROOT)),
        "retrieval_attempts_csv": str(retrieval_csv.relative_to(REPO_ROOT)),
        "candidate_status_csv": str(candidate_status_csv.relative_to(REPO_ROOT)),
        "retrieval_report": str(retrieval_report_path.relative_to(REPO_ROOT)),
        "discovery_pipeline_figure": str(discovery_pipeline_fig.relative_to(REPO_ROOT)),
        "lookup_manifest_figure": str(lookup_manifest_fig.relative_to(REPO_ROOT)),
        "discovery_graph_figure": str(discovery_graph_fig.relative_to(REPO_ROOT)),
        "lookup_vs_verification_figure": str(lookup_vs_verification_fig.relative_to(REPO_ROOT)),
        "discovery_flow_figure": str(discovery_flow_fig.relative_to(REPO_ROOT)),
        "failed_lookup_figure": str(failed_lookup_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "29_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 13. Download artifacts

Run the final cell to package notebook 29 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "29_discovery_artifacts.zip"

notebook_path = NOTEBOOKS / "29_discovery.ipynb"
fallback_notebook_path = Path.cwd() / "29_discovery.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass